# DepoSafety V2 - 3D Gaussian Splatting Pipeline

This notebook processes video to 3D Gaussian Splatting models.

## Features
- Mount Google Drive for input/output
- Extract frames from video
- Run COLMAP for camera pose estimation
- Train 3D Gaussian Splatting model
- Export .ply and .splat files
- Upload to Cloudflare R2
- Notify backend via webhook

## Setup Instructions
1. Set your configuration in the CONFIG cell below
2. Run all cells (Runtime → Run all)
3. Monitor progress in the output

## 1. Configuration

Set these variables before running:

In [ ]:
# ==================== CONFIGURATION ====================

# Input/Output paths in Google Drive
INPUT_VIDEO = "/content/drive/MyDrive/DepoSafety/input/video.mp4"  # Path to input video
OUTPUT_DIR = "/content/drive/MyDrive/DepoSafety/output"  # Output directory

# Processing parameters
FRAME_RATE = 2  # Extract 2 frames per second
RESOLUTION = -1  # -1 for original, or set max width (e.g., 1920)
ITERATIONS = 30000  # Training iterations (7000 for quick test, 30000 for quality)

# Cloudflare R2 Configuration (for model storage)
R2_ACCOUNT_ID = ""  # Your Cloudflare account ID
R2_ACCESS_KEY_ID = ""  # R2 access key
R2_SECRET_ACCESS_KEY = ""  # R2 secret key
R2_BUCKET_NAME = "deposafety-models"  # Bucket name

# Backend Webhook Configuration
WEBHOOK_URL = ""  # Your backend webhook URL
API_KEY = ""  # API key for authentication
JOB_ID = ""  # Leave empty to auto-generate

# =====================================================

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## 3. Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq colmap ffmpeg libgl1-mesa-glx

# Install Python packages
!pip install -q torch torchvision torchaudio
!pip install -q opencv-python pillow tqdm plyfile
!pip install -q boto3 requests

# Clone gaussian-splatting repository
import os
if not os.path.exists('/content/gaussian-splatting'):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /content/gaussian-splatting
    %cd /content/gaussian-splatting
    !pip install -q submodules/diff-gaussian-rasterization
    !pip install -q submodules/simple-knn
    %cd /content

print("Dependencies installed!")

## 4. Define Pipeline Functions

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm

# Generate job ID if not provided
if not JOB_ID:
    JOB_ID = datetime.now().strftime("%Y%m%d_%H%M%S_") + os.urandom(4).hex()
print(f"Job ID: {JOB_ID}")

# Setup directories
WORK_DIR = f"/content/work/{JOB_ID}"
FRAMES_DIR = f"{WORK_DIR}/frames"
COLMAP_DIR = f"{WORK_DIR}/colmap"
GAUSSIAN_DIR = f"{WORK_DIR}/gaussian"
EXPORT_DIR = f"{OUTPUT_DIR}/{JOB_ID}"

for d in [FRAMES_DIR, COLMAP_DIR, GAUSSIAN_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print(f"Export directory: {EXPORT_DIR}")

In [ ]:
def extract_frames(video_path, output_dir, fps=2, resolution=-1):
    """Extract frames from video using ffmpeg"""
    print(f"Extracting frames at {fps} fps...")
    
    cmd = [
        "ffmpeg",
        "-i", video_path,
        "-vf", f"fps={fps}",
        "-q:v", "2",
        f"{output_dir}/frame_%04d.jpg"
    ]
    
    if resolution > 0:
        cmd[3] = f"fps={fps},scale={resolution}:-1"
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        return False
    
    frame_count = len(list(Path(output_dir).glob("*.jpg")))
    print(f"Extracted {frame_count} frames")
    return frame_count > 0

def run_colmap(frames_dir, colmap_dir):
    """Run COLMAP for camera pose estimation"""
    print("Running COLMAP feature extraction...")
    
    db_path = f"{colmap_dir}/database.db"
    
    # Feature extraction
    feat_cmd = [
        "colmap", "feature_extractor",
        "--database_path", db_path,
        "--image_path", frames_dir,
        "--ImageReader.camera_model", "OPENCV",
        "--ImageReader.single_camera", "1"
    ]
    
    result = subprocess.run(feat_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Feature extraction error: {result.stderr}")
        return False
    print("✓ Feature extraction complete")
    
    # Feature matching
    print("Running COLMAP feature matching...")
    match_cmd = [
        "colmap", "exhaustive_matcher",
        "--database_path", db_path
    ]
    
    result = subprocess.run(match_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Feature matching error: {result.stderr}")
        return False
    print("✓ Feature matching complete")
    
    # Sparse reconstruction
    print("Running COLMAP mapper...")
    sparse_dir = f"{colmap_dir}/sparse"
    os.makedirs(sparse_dir, exist_ok=True)
    
    mapper_cmd = [
        "colmap", "mapper",
        "--database_path", db_path,
        "--image_path", frames_dir,
        "--output_path", sparse_dir
    ]
    
    result = subprocess.run(mapper_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Mapper error: {result.stderr}")
        return False
    print("✓ Sparse reconstruction complete")
    
    # Convert to TXT
    txt_dir = f"{colmap_dir}/sparse_txt"
    os.makedirs(txt_dir, exist_ok=True)
    
    converter_cmd = [
        "colmap", "model_converter",
        "--input_path", f"{sparse_dir}/0",
        "--output_path", txt_dir,
        "--output_type", "TXT"
    ]
    
    result = subprocess.run(converter_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Model conversion error: {result.stderr}")
        return False
    print("✓ Model conversion complete")
    
    return True

def prepare_gaussian_data(frames_dir, colmap_dir, gaussian_dir):
    """Prepare data for gaussian-splatting"""
    print("Preparing Gaussian Splatting data...")
    
    input_dir = f"{gaussian_dir}/input"
    os.makedirs(input_dir, exist_ok=True)
    
    # Copy frames
    for frame in Path(frames_dir).glob("*.jpg"):
        shutil.copy(frame, input_dir)
    
    # Copy COLMAP sparse reconstruction
    sparse_src = f"{colmap_dir}/sparse/0"
    sparse_dst = f"{gaussian_dir}/sparse/0"
    os.makedirs(sparse_dst, exist_ok=True)
    
    for f in Path(sparse_src).glob("*"):
        shutil.copy(f, sparse_dst)
    
    print("✓ Data preparation complete")
    return True

print("Functions defined!")

In [ ]:
def train_gaussian_splatting(gaussian_dir, iterations=30000):
    """Train 3D Gaussian Splatting model"""
    print(f"Training Gaussian Splatting model ({iterations} iterations)...")
    
    output_path = f"{gaussian_dir}/output"
    
    train_cmd = [
        sys.executable,
        "/content/gaussian-splatting/train.py",
        "-s", gaussian_dir,
        "-m", output_path,
        "--iterations", str(iterations),
        "--save_iterations", "7000", "30000"
    ]
    
    # Run training
    process = subprocess.Popen(
        train_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    
    for line in process.stdout:
        print(line, end='')
    
    process.wait()
    
    if process.returncode == 0:
        print("✓ Training complete")
        return True
    else:
        print(f"Training failed with code {process.returncode}")
        return False

print("Training function defined!")

In [ ]:
from plyfile import PlyData

def convert_ply_to_splat(ply_path, splat_path):
    """Convert PLY to compressed SPLAT format"""
    print(f"Converting {ply_path} to SPLAT format...")
    
    plydata = PlyData.read(ply_path)
    vertex = plydata['vertex']
    
    x = np.asarray(vertex['x'])
    y = np.asarray(vertex['y'])
    z = np.asarray(vertex['z'])
    
    # SH coefficients for color
    f_dc = np.stack([
        np.asarray(vertex['f_dc_0']),
        np.asarray(vertex['f_dc_1']),
        np.asarray(vertex['f_dc_2'])
    ], axis=1)
    
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))
    
    colors = sigmoid(f_dc) * 255
    colors = colors.astype(np.uint8)
    
    # Opacity
    opacity = sigmoid(np.asarray(vertex['opacity']))
    
    # Scale
    scale = np.stack([
        np.exp(np.asarray(vertex['scale_0'])),
        np.exp(np.asarray(vertex['scale_1'])),
        np.exp(np.asarray(vertex['scale_2']))
    ], axis=1)
    
    # Rotation (quaternion)
    rot = np.stack([
        np.asarray(vertex['rot_0']),
        np.asarray(vertex['rot_1']),
        np.asarray(vertex['rot_2']),
        np.asarray(vertex['rot_3'])
    ], axis=1)
    
    # Normalize quaternions
    rot_norm = np.linalg.norm(rot, axis=1, keepdims=True)
    rot = rot / (rot_norm + 1e-8)
    
    # Pack into SPLAT format
    num_gaussians = len(x)
    splat_data = np.zeros(num_gaussians, dtype=[
        ('position', np.float32, 3),
        ('scale', np.float32, 3),
        ('color', np.uint8, 4),
        ('rotation', np.uint8, 4)
    ])
    
    splat_data['position'] = np.stack([x, y, z], axis=1)
    splat_data['scale'] = scale
    splat_data['color'] = np.concatenate([colors, (opacity * 255).astype(np.uint8)[:, None]], axis=1)
    
    # Pack quaternion into bytes
    rot_uint8 = ((rot + 1) * 127.5).astype(np.uint8)
    splat_data['rotation'] = rot_uint8
    
    # Write SPLAT file
    splat_data.tofile(splat_path)
    
    ply_size = os.path.getsize(ply_path) / (1024 * 1024)
    splat_size = os.path.getsize(splat_path) / (1024 * 1024)
    print(f"✓ SPLAT created: {splat_size:.2f} MB (PLY was {ply_size:.2f} MB, {ply_size/splat_size:.1f}x smaller)")
    
    return True

def export_models(gaussian_dir, export_dir):
    """Export trained models"""
    print("Exporting models...")
    
    model_dir = f"{gaussian_dir}/output/point_cloud"
    
    if not os.path.exists(model_dir):
        print("Error: No trained model found")
        return None
    
    # Find latest iteration
    iter_dirs = list(Path(model_dir).glob("iteration_*"))
    if not iter_dirs:
        print("Error: No iteration folders found")
        return None
    
    latest_iter = sorted(iter_dirs)[-1]
    ply_file = latest_iter / "point_cloud.ply"
    
    if not ply_file.exists():
        print("Error: PLY file not found")
        return None
    
    # Copy PLY
    ply_dst = f"{export_dir}/model.ply"
    shutil.copy(ply_file, ply_dst)
    print(f"✓ Exported PLY: {ply_dst}")
    
    # Convert to SPLAT
    splat_dst = f"{export_dir}/model.splat"
    convert_ply_to_splat(ply_file, splat_dst)
    
    return {
        'ply': ply_dst,
        'splat': splat_dst,
        'iteration': latest_iter.name
    }

print("Export functions defined!")

In [ ]:
import boto3
import requests

def upload_to_r2(file_path, remote_key, content_type=None):
    """Upload file to Cloudflare R2"""
    if not all([R2_ACCOUNT_ID, R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_BUCKET_NAME]):
        print("R2 credentials not configured, skipping upload")
        return None
    
    try:
        s3 = boto3.client(
            's3',
            endpoint_url=f'https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com',
            aws_access_key_id=R2_ACCESS_KEY_ID,
            aws_secret_access_key=R2_SECRET_ACCESS_KEY
        )
        
        extra_args = {}
        if content_type:
            extra_args['ContentType'] = content_type
        
        s3.upload_file(file_path, R2_BUCKET_NAME, remote_key, ExtraArgs=extra_args)
        
        public_url = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com/{R2_BUCKET_NAME}/{remote_key}"
        print(f"✓ Uploaded to R2: {public_url}")
        return public_url
        
    except Exception as e:
        print(f"R2 upload failed: {e}")
        return None

def notify_webhook(status, models=None, error=None, r2_urls=None):
    """Notify backend via webhook"""
    if not WEBHOOK_URL:
        print("Webhook URL not configured, skipping notification")
        return
    
    payload = {
        'event': 'processing_complete',
        'job_id': JOB_ID,
        'status': status,
        'timestamp': datetime.now().isoformat(),
        'models': models or {},
        'r2_urls': r2_urls or {}
    }
    
    if error:
        payload['error'] = error
    
    headers = {
        'Content-Type': 'application/json',
        'User-Agent': 'DepoSafety-ML-Pipeline/1.0'
    }
    
    if API_KEY:
        headers['Authorization'] = f'Bearer {API_KEY}'
        headers['X-API-Key'] = API_KEY
    
    try:
        response = requests.post(WEBHOOK_URL, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
        print(f"✓ Webhook notification sent: {response.status_code}")
    except Exception as e:
        print(f"Webhook notification failed: {e}")

print("Upload and notification functions defined!")

## 5. Run Pipeline

In [ ]:
# Verify input video exists
if not os.path.exists(INPUT_VIDEO):
    raise FileNotFoundError(f"Input video not found: {INPUT_VIDEO}")

print(f"Input video: {INPUT_VIDEO}")
print(f"Output directory: {EXPORT_DIR}")
print(f"Job ID: {JOB_ID}")
print("\n" + "="*50)
print("Starting 3D Gaussian Splatting Pipeline")
print("="*50 + "\n")

In [ ]:
# Step 1: Extract frames
success = extract_frames(INPUT_VIDEO, FRAMES_DIR, FRAME_RATE, RESOLUTION)
if not success:
    notify_webhook("failed", error="Frame extraction failed")
    raise RuntimeError("Frame extraction failed")

In [ ]:
# Step 2: Run COLMAP
success = run_colmap(FRAMES_DIR, COLMAP_DIR)
if not success:
    notify_webhook("failed", error="COLMAP failed")
    raise RuntimeError("COLMAP failed")

In [ ]:
# Step 3: Prepare data
success = prepare_gaussian_data(FRAMES_DIR, COLMAP_DIR, GAUSSIAN_DIR)
if not success:
    notify_webhook("failed", error="Data preparation failed")
    raise RuntimeError("Data preparation failed")

In [ ]:
# Step 4: Train Gaussian Splatting
success = train_gaussian_splatting(GAUSSIAN_DIR, ITERATIONS)
if not success:
    notify_webhook("failed", error="Training failed")
    raise RuntimeError("Training failed")

In [ ]:
# Step 5: Export models
models = export_models(GAUSSIAN_DIR, EXPORT_DIR)
if not models:
    notify_webhook("failed", error="Export failed")
    raise RuntimeError("Export failed")

print(f"\nModels exported to: {EXPORT_DIR}")
print(f"  - PLY: {models['ply']}")
print(f"  - SPLAT: {models['splat']}")

In [ ]:
# Step 6: Upload to R2
r2_urls = {}

if all([R2_ACCOUNT_ID, R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_BUCKET_NAME]):
    print("\nUploading to Cloudflare R2...")
    
    ply_url = upload_to_r2(
        models['ply'],
        f"models/{JOB_ID}/model.ply",
        'application/octet-stream'
    )
    if ply_url:
        r2_urls['ply'] = ply_url
    
    splat_url = upload_to_r2(
        models['splat'],
        f"models/{JOB_ID}/model.splat",
        'application/octet-stream'
    )
    if splat_url:
        r2_urls['splat'] = splat_url
else:
    print("\nR2 credentials not configured, skipping upload")

In [ ]:
# Step 7: Notify webhook
print("\nSending webhook notification...")
notify_webhook("completed", models=models, r2_urls=r2_urls)

print("\n" + "="*50)
print("Pipeline Complete!")
print("="*50)
print(f"Job ID: {JOB_ID}")
print(f"Output: {EXPORT_DIR}")
if r2_urls:
    print(f"R2 URLs: {r2_urls}")

## 6. Cleanup (Optional)

Run this cell to clean up temporary files and free disk space:

In [ ]:
# Clean up working directory (keeps exported models in Drive)
import shutil
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    print(f"Cleaned up: {WORK_DIR}")

# Check disk usage
!df -h /content

## Troubleshooting

### Out of Memory
- Reduce `RESOLUTION` (e.g., 1280 or 1024)
- Reduce `ITERATIONS` (e.g., 7000)
- Use a shorter video segment

### COLMAP fails
- Ensure video has good texture/features
- Increase `FRAME_RATE` for more frames
- Check that frames were extracted successfully

### Training fails
- Check GPU memory: `!nvidia-smi`
- Try reducing batch size (modify train.py parameters)

### R2 Upload fails
- Verify credentials are correct
- Check bucket exists and is accessible
- Models are still saved to Google Drive even if upload fails